# SPINE — Colab quickstart

Two-phase post-training (QLoRA SFT → DPO) for strict JSON extraction from messy
network-operations text.

**Runtime → Change runtime type → GPU.** A T4 is enough with the 1.5B config;
an A100 is needed for the Qwen3-8B config.


## 0 · Setup


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('bf16 supported:', torch.cuda.is_bf16_supported())


In [ ]:
# !git clone https://github.com/NiteeshSK/spine.git
%cd spine
!pip install -q -r requirements.txt


In [ ]:
import os
os.environ['PYTHONPATH'] = 'src'
# T4 users: use the 1.5B configs. A100 users: drop the _t4 suffix.
SFT_CFG = 'configs/sft_qlora_t4.yaml'
DPO_CFG = 'configs/dpo_t4.yaml'
BASE    = 'Qwen/Qwen2.5-1.5B-Instruct'


## 1 · Data — generate, audit, verify

The audit gate exits non-zero on schema violations, ungrounded spans, duplicates
or cross-split leakage. If this cell fails, do not train.


In [ ]:
!python -m spine.data.synth --out data/raw
!python -m spine.data.build_sft --raw data/raw --out data/sft
!cp data/sft/pool.jsonl data/sft/dpo_pool.jsonl
!python -m pytest -q


In [ ]:
import json
row = json.loads(open('data/sft/test.jsonl').readline())
print(row['text'][:600]); print('\n--- gold ---\n'); print(json.dumps(json.loads(row['target']), indent=2))


## 2 · Baseline first

Always measure the prompted baseline *before* training. Without it there is no
gap to report, and the honest comparison is the whole deliverable.


In [ ]:
!python -m spine.eval.run_eval --split test --limit 200 \
    --model $BASE --prompt baseline --tag base_prompted --batch-size 4


## 3 · Phase 1 — QLoRA SFT


In [ ]:
!python -m spine.train.sft --config $SFT_CFG


In [ ]:
from spine.viz import plots
from pathlib import Path
import yaml
out = Path(yaml.safe_load(open(SFT_CFG))['output_dir'])
plots.plot_training_curves(out / 'train_log.csv', Path('assets/sft_curves.png'))
from IPython.display import Image; Image('assets/sft_curves.png')


In [ ]:
ADAPTER = f'{yaml.safe_load(open(SFT_CFG))["output_dir"]}/adapter'
!python -m spine.eval.run_eval --split test --model $BASE --adapter $ADAPTER --tag sft --batch-size 4


## 4 · Phase 2 — mine preference pairs, then DPO

Check the reported **yield** and **failure mix** before training. A yield near 0%
or near 100% means the margin gates are miscalibrated.


In [ ]:
!python -m spine.data.build_dpo --model $BASE --adapter $ADAPTER \
    --pool data/sft/dpo_pool.jsonl --limit 600 --k 6 --temperature 0.9 --batch-size 4


In [ ]:
!python -m spine.train.dpo --config $DPO_CFG


In [ ]:
DPO_ADAPTER = f'{yaml.safe_load(open(DPO_CFG))["output_dir"]}/adapter'
!python -m spine.eval.run_eval --split test --model $BASE --adapter $DPO_ADAPTER --tag dpo --batch-size 4
!python -m spine.eval.run_eval --split ood  --model $BASE --adapter $DPO_ADAPTER --tag dpo --batch-size 4


## 5 · Report

Figures and tables are generated from the artifacts in `reports/` and `runs/`.
Paste the table into `docs/RESULTS.md` and commit `reports/` + `assets/`.


In [ ]:
!python -m spine.viz.plots --sft-log {out}/train_log.csv --out assets
!python -m spine.eval.compare --split test


### Save the adapter

LoRA adapters are tens of MB, not gigabytes — they are cheap to keep and to share.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!cp -r {ADAPTER} /content/drive/MyDrive/spine_sft_adapter
!cp -r {DPO_ADAPTER} /content/drive/MyDrive/spine_dpo_adapter
